# Project Overview & Objectives

 This research document outlines a production-grade script for **Domain-Specific Pretraining and Fine-Tuning** of Large Language Models (LLMs). The primary objective is to adapt a pre-trained base model to a specialized medical/pharmaceutical domain using unsupervised causal language modeling (next-token prediction), entirely without relying on instruction-response pairs.
 
**Key Phases in this Architecture:**

**Environment Setup:** Installing and configuring required libraries.

**Data Acquisition & EDA:** Fetching prebuilt data for testing and parsing custom raw PDF documents.

**Preprocessing & Feature Engineering:** Chunking texts into context windows and preparing causal language modeling targets.

**Model Development & Training:** Exploring Full Fine-Tuning, Layer Freezing, and Parameter-Efficient Fine-Tuning (LoRA).

**Evaluation & Results:** Generating text to verify domain adaptation.

**Conclusion & Future Work:** Mapping out instruction tuning and preference alignment.

# Environment Setup

## Installation of Required Libraries

In [1]:
%pip install -U peft bitsandbytes transformers accelerate trl PyMuPDF datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 113.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 78.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transform

## Importing Libraries

In [18]:
from IPython.display import display
from datasets import load_dataset, Dataset
import fitz
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
import os
import re
from peft import LoraConfig, get_peft_model, TaskType
import torch
import gc

## Mount Google Drive and create a target folder for notebook assets

In [3]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ASSETS = Path('/content/drive/MyDrive/LLM-Fine-Tuning/DomainSpecific/assets')
DRIVE_ASSETS.mkdir(parents=True, exist_ok=True)
print("Drive assets folder:", DRIVE_ASSETS)

Mounted at /content/drive
Drive assets folder: /content/drive/MyDrive/LLM-Fine-Tuning/DomainSpecific/assets


# Data Acquisition & Exploratory Data Analysis (EDA)

## Ingesting Prebuilt Demonstration Data from HuggingFace

In [4]:
dataset = load_dataset("roneneldan/TinyStories", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [5]:
display(dataset[:3])

{'text': ['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.',
  'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leav

## Custom Extraction Pipeline for Domain-Specific PDFs

In [6]:
def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

In [7]:
pdf_texts = extract_text_from_pdf(f"{DRIVE_ASSETS}/Metformin.pdf")
display(pdf_texts[:3])

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis. \n \nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA red

# Preprocessing & Feature Engineering

## Text Segmentation and Cleaning

In [8]:
def split_paragraphs(pages):
    """
    Uses regex to split page text on double line breaks.
    Filters out extremely short, fragmented lines (e.g., page numbers or headers).
    """
    processed_paragraphs = []
    for page in pages:
        chunks = re.split(r'\n\s*\n', page)
        for chunk in chunks:
            clean_chunk = chunk.strip()
            if len(clean_chunk) > 50:  # Filter out very short lines
                processed_paragraphs.append(clean_chunk)
    return processed_paragraphs

In [9]:
domain_paragraphs = split_paragraphs(pdf_texts)
formatted_data = [{"text": para} for para in domain_paragraphs]
domain_dataset = Dataset.from_list(formatted_data)
display(domain_dataset[:3])

{'text': ['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.',
  'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HM

## Mapping the Tokenizer

In [11]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# Initialize tokenizer and handle padding constraints
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [10]:
def tokenize_fn(examples):
    """
    Tokenizes raw text strings into dense integer IDs, truncating 
    and padding to standard 512 context length.
    """
    tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
    # Duplicate input_ids into labels for auto-shifted next-token prediction
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [12]:
# Map tokenization function across the entire domain dataset

tokenized_dataset = domain_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

## Dynamic Data Collator Injection

In [14]:
# Instantiating the proper language modeling collator. mlm=False means Causal (GPT-style) LM

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Model Development & Training

In [16]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

### Approach 1: Full Fine-Tuning (Resource Intensive)
 This updates all 1.1 Billion weights. While mathematically optimal, it is highly prone to catastrophic forgetting and requires immense VRAM.

In [ ]:
training_args_full = TrainingArguments(
    output_dir="./llama-pharma-domain-full",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

trainer_full = Trainer(
    model=model,
    args=training_args_full,
    train_dataset=tokenized_dataset,
    data_collator=data_collator # Added for structural completeness
)

In [ ]:
# trainer_full.train()

### Approach 2: Partial Fine-Tuning (Layer Freezing)
This methodology locks early feature layers and only updates the final few transformer blocks along with the language model head to save computational overhead.

In [ ]:
# Step A: Freeze everything first
for param in model.parameters():
    param.requires_grad = False

In [ ]:
# Step B: Dynamically find how many layers the model has
num_layers = model.config.num_hidden_layers
unfreeze_last_n_layers = 4
start_layer = max(0, num_layers - unfreeze_last_n_layers)

print(f"Total layers: {num_layers}. Unfreezing layers {start_layer} to {num_layers - 1}...")

In [ ]:
# Step C: Unfreeze the targeted layers, the final LayerNorm, and the LM Head
for name, param in model.named_parameters():
    # 1. Unfreeze the last N transformer blocks
    if any(f"model.layers.{i}." in name for i in range(start_layer, num_layers)):
        param.requires_grad = True
    
    # 2. Unfreeze the final layer normalization (crucial for stability)
    if "model.norm" in name:
        param.requires_grad = True
        
    # 3. Unfreeze the Language Modeling Head (the output projection)
    if "lm_head" in name:
        param.requires_grad = True

In [ ]:
# Verification: Print trainable percentage
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Trainable Parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

In [ ]:
# Training Configuration & Execution

# Enable gradient checkpointing to save memory if your GPU is small
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir="./tinyllama-pharma-frozen-layers",
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,  # We can use a slightly higher LR than full fine-tuning
    bf16=True,  # Change to fp16=True if your GPU doesn't support bfloat16
    logging_steps=10,
    save_total_limit=1,
    report_to="none"
)

trainer_partial = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [ ]:
# trainer_partial.train()

### Approach 3: Parameter-Efficient Fine-Tuning (LoRA)
Low-Rank Adaptation (LoRA) injects trainable rank decomposition matrices into the transformer blocks while keeping the base model weights frozen. This dramatically cuts memory requirements.


In [ ]:
# Clean up VRAM allocations before starting quantized loading
# del model
# gc.collect()
# torch.cuda.empty_cache()

# 1. Define the quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    # Optional but recommended for training:
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

# 2. Load the model using the config
peft_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Configure LoRA Adapter
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                  
    lora_alpha=16,        
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

# wrap the model with get_peft_model so it targets weights correctly!
non_inst_model_lora = get_peft_model(peft_model, lora_config)
non_inst_model_lora.print_trainable_parameters() # Good practice to verify LoRA is active

lora_training_args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,  
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

# add data_collator and pass the actual wrapped LoRA model
trainer = Trainer(
    model=non_inst_model_lora,
    args=lora_training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


TrainOutput(global_step=5, training_loss=1.7329509735107422, metrics={'train_runtime': 12.7829, 'train_samples_per_second': 1.565, 'train_steps_per_second': 0.391, 'total_flos': 63629646888960.0, 'train_loss': 1.7329509735107422, 'epoch': 5.0})

# Evaluation & Results

In [ ]:
eval_model = non_inst_model_lora  # active in-memory representation

# Setup prompt validation
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Text generation execution
outputs = eval_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")



--- Model Generated Output ---

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe was superior to Ezetimibe monotherapy in reducing the risk of cardiovascular disease (CVD) events. The most common adverse reactions with Atorvastatin 80 mg once daily are headache, back pain, chest pain, flatulence, nausea, and diarrhea.
Sandoz is a division of Novartis AG and is the pharmaceutical division of Sandoz Inc., which


In [22]:
print("\n--- Model Generated Output ---\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=False))


--- Model Generated Output ---

<s> Clinical trials demonstrated that combining Atorvastatin with Ezetimibe was superior to Ezetimibe monotherapy in reducing the risk of cardiovascular disease (CVD) events. The most common adverse reactions with Atorvastatin 80 mg once daily are headache, back pain, chest pain, flatulence, nausea, and diarrhea.
Sandoz is a division of Novartis AG and is the pharmaceutical division of Sandoz Inc., which
